# Detection methods, from the raw dataEach of the detectors the viewer runs is a short algorithm. This notebook writes them out again**from scratch** — plain numpy and scipy, no import from `dataanalyzer` except the loader — so aresult you have looked at in the viewer can be checked against an independent implementation, andso the method itself is easy to read, change and argue with.Three methods, as published:| | method ||---|---|| **Ripples** | LFP at 1250 Hz, 120–200 Hz (Butterworth order 3), squared, smoothed, normalised to its own mean and SD: the *normalised squared signal* (NSS). Peak over 5 SD, start and end at 2 SD, duration 15–100 ms. || **Dentate spikes** | LFP 1–200 Hz (Butterworth order 4), peaks over 6 × the median absolute amplitude of that signal, events at least 50 ms apart. || **High theta** | 5–12 Hz power from a multitaper spectrogram (4 s window, 2 s step) on a reference channel, smoothed, and the bins above the 90th percentile of the whole session. |The last section compares each one against the result saved by the viewer, event by event.

In [ ]:
from pathlib import Pathimport numpy as np, jsonimport matplotlib.pyplot as pltfrom scipy.signal import butter, sosfiltfilt, windowsimport sys; sys.path.insert(0, str(Path.cwd().parent))from dataanalyzer import io as dio          # only to read the Open Ephys files# --- the recording to check. Point this at the one you were looking at in the viewer.RECORDING = "/Volumes/Extreme/RippleDetection/D4/Oct062026_StimTestAndRecord/2026-10-06_15-29-14/Record Node 110"STREAM    = "BroadbandStream-A"ts = dio.load(RECORDING, stream=STREAM)print(ts.source.split("/")[-4], "|", ts.stream, "|", f"{ts.duration:.0f} s", "|", ts.n_channels, "channels", "|", f"{ts.fs:g} Hz")electrodes = ts.headstage_channels()print(len(electrodes), "electrodes:", electrodes[:6], "...")

## Reading the signalThe recording is a memory-mapped int16 file, so nothing is read until asked for. Everything belowworks on one or more channels decimated to ~1250 Hz, which is all these methods need and makes aten-minute recording a few seconds of work.

In [ ]:
TARGET_HZ = 1250.0def load_channel(name, rate_hz=TARGET_HZ, t0=None, t1=None):    """One channel in µV, anti-aliased and decimated to about rate_hz. Returns (t, y, fs)."""    fs = float(ts.fs)    step = max(1, int(fs // rate_hz))    lo = 0 if t0 is None else int(np.searchsorted(ts.t, t0))    hi = ts.n if t1 is None else int(np.searchsorted(ts.t, t1))    y = ts.block(lo, hi, [name])[:, 0].astype(float)    if step > 1:        sos = butter(4, (rate_hz / 2.5), btype="low", fs=fs, output="sos")        y = sosfiltfilt(sos, y)[::step]    return ts.t[lo:hi:step], y, fs / stepdef bandpass(y, fs, low, high, order=4):    """Zero-phase Butterworth band-pass (filtfilt, so no phase shift)."""    sos = butter(order, [low, high], btype="band", fs=fs, output="sos")    return sosfiltfilt(sos, y)t, y, fs = load_channel(electrodes[0])print(f"{len(y):,} samples at {fs:g} Hz, {y.std():.0f} µV SD")

## 1 · Ripples — normalised squared signal> the broadband LFP (1250 Hz) was filtered in the 120–200 Hz band (Butterworth; order = 3) and> transformed to a normalised square signal (NSS). Ripple peaks were identified by the signal> exceeding 5 standard deviations above the mean NSS. The start and end timestamps of the ripple> were defined by a threshold of 2 standard deviations above the mean NSS. Ripple duration limits> were between 15 ms and 100 ms.The one detail the text leaves out is the smoothing of the squared signal; `FindRipples` uses an11-sample boxcar at 1250 Hz (8.8 ms), which is what both the viewer and this notebook use.Without it, the squared signal is spiky and the 2 SD crossings that set the edges jitter.

In [ ]:
def nss(y, fs, low=120.0, high=200.0, order=3, smooth_ms=8.8):    """The normalised squared signal, and the mean/SD it was normalised with."""    filt = bandpass(y, fs, low, high, order)    sq = filt ** 2    win = int(round(smooth_ms / 1000 * fs))    if win > 1:                                     # boxcar, via a cumulative sum        c = np.concatenate([[0.0], np.cumsum(sq)])        half = win // 2        idx = np.arange(sq.size)        a = np.clip(idx - half, 0, sq.size); b = np.clip(idx - half + win, 0, sq.size)        sq = (c[b] - c[a]) / np.maximum(b - a, 1)    mean, sd = sq.mean(), sq.std()    return (sq - mean) / sd, mean, sddef find_ripples(t, z, high_sd=5.0, low_sd=2.0, min_ms=15.0, max_ms=100.0, merge_ms=30.0):    """Runs over low_sd that peak over high_sd, merged if close, then the duration limits."""    above = z > low_sd    edges = np.diff(above.astype(np.int8))    starts = np.flatnonzero(edges == 1) + 1    ends = np.flatnonzero(edges == -1) + 1    if above[0]: starts = np.insert(starts, 0, 0)    if above[-1]: ends = np.append(ends, above.size)    runs = []    for a, b in zip(starts, ends):        if runs and t[a] - t[runs[-1][1] - 1] < merge_ms / 1000:            runs[-1][1] = b        else:            runs.append([a, b])    kept, rejected = [], []    for a, b in runs:        k = a + int(np.argmax(z[a:b]))        if z[k] < high_sd:            continue        e = dict(start=t[a], end=t[b - 1], peak=t[k], score=z[k], duration=t[b - 1] - t[a])        (kept if min_ms / 1000 <= e["duration"] <= max_ms / 1000 else rejected).append(e)    return kept, rejected# which channel? the one whose ripple band carries the most powert_all, X, fs_d = None, {}, Nonepower = {}for name in electrodes:    tt, yy, fs_d = load_channel(name)    t_all = tt    X[name] = yy    power[name] = bandpass(yy, fs_d, 120, 200, 3).std()ripple_channel = max(power, key=power.get)print("ripple channel:", ripple_channel, f"({power[ripple_channel]:.1f} µV in band)")z, mean, sd = nss(X[ripple_channel], fs_d)ripples, too_long = find_ripples(t_all, z)print(f"{len(ripples)} ripples, {len(too_long)} rejected on duration — "      f"{len(ripples) / ts.duration * 60:.1f} per minute")

In [ ]:
# what one looks like, and the distribution of the measureif ripples:    e = max(ripples, key=lambda e: e["score"])    k = (t_all > e["peak"] - 0.15) & (t_all < e["peak"] + 0.15)    fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)    ax[0].plot(t_all[k], X[ripple_channel][k], lw=0.6, color="0.4"); ax[0].set_ylabel("µV, raw")    ax[1].plot(t_all[k], bandpass(X[ripple_channel], fs_d, 120, 200, 3)[k], lw=0.8); ax[1].set_ylabel("120–200 Hz")    ax[2].plot(t_all[k], z[k], lw=1); ax[2].axhline(5, ls="--", c="C3"); ax[2].axhline(2, ls=":", c="C3")    ax[2].set_ylabel("NSS (SD)"); ax[2].set_xlabel("s")    for a in ax: a.axvspan(e["start"], e["end"], color="C1", alpha=0.15)    fig.suptitle(f"largest ripple: {e['peak']:.3f} s, {e['score']:.1f} SD, {e['duration']*1000:.0f} ms")    plt.tight_layout()fig, ax = plt.subplots(1, 2, figsize=(10, 3))ax[0].hist([e["duration"] * 1000 for e in ripples], bins=20); ax[0].set_xlabel("duration (ms)")ax[1].hist([e["score"] for e in ripples], bins=20); ax[1].set_xlabel("peak NSS (SD)")plt.tight_layout()

## 2 · Dentate spikes> We detected the peaks of the filtered LFP (1–200 Hz, Butterworth; order = 4) in the channels> covering the dentate gyrus (DG). A threshold of 6 times the median absolute amplitude of the> filtered LFP were employed, and we only considered events that were at least 50 ms apart.Dentate spikes are **positive**, so the sign matters; the depth profile below is what says whetherthe events really come from the dentate rather than being a sharp wave or an artefact.

In [ ]:
def dentate_spikes(t, y, fs, low=1.0, high=200.0, order=4, factor=6.0, refractory_ms=50.0, sign=1):    filt = bandpass(y, fs, low, high, order)    threshold = factor * np.median(np.abs(filt))    v = sign * filt    above = v > threshold    if not above.any():        return [], threshold, filt    edges = np.diff(above.astype(np.int8))    starts = np.flatnonzero(edges == 1) + 1    ends = np.flatnonzero(edges == -1) + 1    if above[0]: starts = np.insert(starts, 0, 0)    if above[-1]: ends = np.append(ends, above.size)    cand = []    for a, b in zip(starts, ends):        k = a + int(np.argmax(v[a:b]))        half = v[k] / 2        lo, hi = k, k        while lo > 0 and v[lo - 1] > half: lo -= 1        while hi < v.size - 1 and v[hi + 1] > half: hi += 1        cand.append(dict(peak=t[k], start=t[lo], end=t[hi], amp_uV=v[k], score=v[k] / threshold))    cand.sort(key=lambda e: -e["score"])              # the largest of a close group wins    kept = []    for e in cand:        if all(abs(e["peak"] - o["peak"]) >= refractory_ms / 1000 for o in kept):            kept.append(e)    return sorted(kept, key=lambda e: e["peak"]), threshold, filtbest = Nonefor name in electrodes:                                # the contact with the largest positive peaks    ev, thr, _ = dentate_spikes(t_all, X[name], fs_d)    if ev and (best is None or np.mean([e["amp_uV"] for e in ev]) > best[1]):        best = (name, float(np.mean([e["amp_uV"] for e in ev])), ev, thr)if best:    ds_channel, _, spikes_ds, ds_thr = best    print(f"{len(spikes_ds)} dentate spikes on {ds_channel}, threshold {ds_thr:.0f} µV "          f"({len(spikes_ds) / ts.duration * 60:.1f} per minute)")else:    ds_channel, spikes_ds, ds_thr = electrodes[0], [], 0.0    print("no events over threshold on any contact")

In [ ]:
# the depth profile: where across the probe are these events largest?if spikes_ds:    idx = np.searchsorted(t_all, [e["peak"] for e in spikes_ds])    prof = np.array([[bandpass(X[n], fs_d, 1, 200, 4)[i] for i in idx] for n in electrodes]).mean(axis=1)    fig, ax = plt.subplots(1, 2, figsize=(10, 4))    ax[0].plot(prof, np.arange(len(electrodes)), "o-")    ax[0].axvline(0, c="0.7"); ax[0].invert_yaxis()    ax[0].set_yticks(range(len(electrodes))); ax[0].set_yticklabels(electrodes, fontsize=6)    ax[0].set_xlabel("mean amplitude at the event (µV)"); ax[0].set_title("depth profile")    e = max(spikes_ds, key=lambda e: e["amp_uV"])    k = (t_all > e["peak"] - 0.1) & (t_all < e["peak"] + 0.1)    for n in electrodes[::4]:        ax[1].plot(t_all[k], bandpass(X[n], fs_d, 1, 200, 4)[k], lw=0.8, label=n)    ax[1].axvline(e["peak"], c="0.7", ls="--"); ax[1].legend(fontsize=6); ax[1].set_xlabel("s")    ax[1].set_title(f"largest event, {e['amp_uV']:.0f} µV")    plt.tight_layout()

## 3 · High theta — multitaper spectrogram, 90th percentile> LFP signals were filtered in the theta band (5–12 Hz) using a zero-phase bandpass filter. Theta> power was computed using multitaper spectrogram (window: 4 s, step: 2 s) from a reference channel> located in the stratum oriens, where theta amplitude is typically high and stable. The> instantaneous theta power time series was smoothed and thresholded, to define high theta periods.> Specifically, time bins in which theta power exceeded 90th percentile of the full session theta> power distribution was labeled as high theta.A percentile threshold labels a fixed *share* of the session — about a tenth of it — whatever theanimal was doing. It answers "when was theta strongest here", not "was there theta at all".

In [ ]:
def multitaper(y, fs, win_s=4.0, step_s=2.0, nw=3.0):    """(bin centres, frequencies, power): Slepian tapers averaged, one spectrum per window."""    n, step = int(round(win_s * fs)), int(round(step_s * fs))    tapers = windows.dpss(n, nw, int(2 * nw - 1))    starts = np.arange(0, y.size - n + 1, step)    f = np.fft.rfftfreq(n, 1 / fs)    out = np.empty((starts.size, f.size))    for i, s in enumerate(starts):        seg = y[s:s + n] - y[s:s + n].mean()        out[i] = (np.abs(np.fft.rfft(tapers * seg, axis=-1)) ** 2).mean(axis=0) * (2 / fs)    return (starts + n / 2) / fs, f, outdef high_theta(t, y, fs, low=5.0, high=12.0, win_s=4.0, step_s=2.0, smooth_s=4.0, pct=90.0):    tc, f, spec = multitaper(y, fs, win_s, step_s)    band = (f >= low) & (f <= high)    p = np.trapezoid(spec[:, band], f[band], axis=1)    n = int(round(smooth_s / step_s))    if n > 1:        p = np.convolve(p, np.ones(n) / n, mode="same")    threshold = np.percentile(p, pct)    tc = tc + float(t[0])    hits = p > threshold    epochs, half = [], step_s / 2    for i in np.flatnonzero(hits):        if epochs and tc[i] - epochs[-1]["end"] <= half * 2 + 1e-9:            epochs[-1]["end"] = tc[i] + half        else:            epochs.append(dict(start=tc[i] - half, end=tc[i] + half))    return epochs, tc, p, thresholdtheta_power = {n: np.median(np.abs(bandpass(X[n], fs_d, 5, 12))) for n in electrodes}theta_channel = max(theta_power, key=theta_power.get)epochs, tc, tp, thr = high_theta(t_all, X[theta_channel], fs_d)covered = sum(e["end"] - e["start"] for e in epochs)print(f"reference {theta_channel}: {len(epochs)} high-theta epochs, {covered:.0f} s "      f"({covered / ts.duration * 100:.0f} % of the session)")fig, ax = plt.subplots(figsize=(11, 3))ax.plot(tc, tp, lw=0.8)ax.axhline(thr, ls="--", c="C3", label=f"90th percentile")for e in epochs: ax.axvspan(e["start"], e["end"], color="C1", alpha=0.25)ax.set_xlabel("s"); ax.set_ylabel("5–12 Hz power"); ax.legend(); plt.tight_layout()

## 4 · Against what the viewer savedThe viewer writes every run to `<recording>/dataanalyzer/[<stream>/]events/<name>.json`. If thetwo implementations agree, nearly every event of one is within a few milliseconds of an event ofthe other; the ones that are not are worth looking at in the Events tab.

In [ ]:
def saved(name):    d = Path(ts.source) / "dataanalyzer"    if ts.results_subdir:        d = d / ts.results_subdir    p = d / "events" / f"{name}.json"    if not p.exists():        print(f"no saved {name} — run it in the viewer first ({p})")        return None    return json.loads(p.read_text())def agree(mine, theirs, tol_ms=25.0, label=""):    a = np.array(sorted(e["peak"] for e in mine))    b = np.array(sorted(e["peak"] for e in theirs))    if not a.size or not b.size:        print(f"{label}: {a.size} here, {b.size} saved — nothing to compare")        return    near = lambda x, pool: np.min(np.abs(pool - x)) if pool.size else np.inf    matched = sum(near(x, b) <= tol_ms / 1000 for x in a)    print(f"{label}: {a.size} here, {b.size} saved, {matched} within {tol_ms:g} ms "          f"({matched / a.size * 100:.0f} % of mine, {matched / b.size * 100:.0f} % of theirs)")    extra = [round(float(x), 3) for x in a if near(x, b) > tol_ms / 1000][:10]    missing = [round(float(x), 3) for x in b if near(x, a) > tol_ms / 1000][:10]    if extra:   print("   only here :", extra)    if missing: print("   only saved:", missing)r = saved("swr_nss")if r: agree(ripples, r["events"], label=f"ripples (saved on {r['channel']}, mine on {ripple_channel})")r = saved("dentate_spikes")if r: agree(spikes_ds, r["events"], label=f"dentate spikes (saved on {r['channel']}, mine on {ds_channel})")r = saved("theta_mt")if r:    mine_s = sum(e["end"] - e["start"] for e in epochs)    theirs_s = sum(e["end"] - e["start"] for e in r["events"])    print(f"high theta: {len(epochs)} epochs / {mine_s:.0f} s here, "          f"{len(r['events'])} / {theirs_s:.0f} s saved (on {r['channel']}, mine on {theta_channel})")

### If they disagree* **A different channel.** Each method picks its own channel when asked for `auto`; the printout  above says which. Pass the same name to both before concluding anything.* **Time ranges.** The Events tab can restrict a set to parts of the recording; the saved file  still holds every event, but what the tab counts is what lies inside the ranges.* **Merged pulses, rejected events.** `events` are the kept ones; `rejected` holds the rest with a  reason, and is worth reading before trusting a count.* **Edges.** Decimating and filtering treat the first and last second differently here and there;  events within a second of either end are not worth arguing about.